# 03 — Model

**Project:** Do U.S. climate change concerns move European green-minus-brown returns?
**Author:** Rafael Machado Cerqueira — Advanced Data Analytics, HEC Lausanne, Autumn 2026

**Input:** `data/processed/` (from `02_clean.ipynb`). **Output:** tables in `results/` (used by `04_figures.ipynb` and the paper).

| Step | What it does |
|---|---|
| 1 | Load the cleaned panels |
| 2 | Build **concern shocks**: the part of the MCCC index that its own past does not predict |
| 2b | Improved shocks (weekday, calendar-day and trend adjustment; 20-day moving-average shock) and the choice of the main definition |
| 3 | Timing: same-day and previous-day shocks (U.S. news often arrives after the European close) |
| 4 | Regressions: U.S. replication, then Europe (Newey-West standard errors) |
| 4b–4c | Regressions with each shock definition; U.S. on the Ardia et al. sample period |
| 5 | Check the assumptions of the linear model |
| 6 | Robustness checks |
| 6b–6c | Longer horizons (5-day, weekly); large shocks (exploratory) |
| 7 | Non-linear model: random forest vs linear regression, out of sample, time-ordered |
| 8 | Save the results |

*Code drafted with Claude (Anthropic) and reviewed by the author — declared in the paper's appendix.*

## 0. Setup and parameters

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LinearRegression
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.stattools import jarque_bera
from statsmodels.tsa.stattools import adfuller

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW, PROC, RES = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "results"
RES.mkdir(exist_ok=True)

# ---- Modelling choices (each one is reported in the paper) ----
SEED = 42
PRESAMPLE_END = "2009-12-31"   # the AR model of expected concern is first estimated on 2003-2009
AR_MAX_LAGS = 10               # candidate lag orders 1..10 trading days
AR_CRITERION = "bic"           # lag order chosen by BIC on the pre-sample
NW_LAGS = 5                    # Newey-West lags (one trading week)
WINSOR_Q = 0.01                # robustness: clip variables at the 1st and 99th percentiles
PARIS_SPLIT = "2016-01-01"     # robustness: before / after the Paris Agreement (Dec 2015)
OOS_FIRST_YEAR = 2015          # out-of-sample test years: 2015-2025, trained on all earlier years
RF_PARAMS = dict(n_estimators=300, max_depth=4, min_samples_leaf=50, max_features=0.5,
                 random_state=SEED, n_jobs=-1)  # shallow trees, large leaves: daily returns are very noisy

clean_log = json.loads((PROC / "cleaning_log.json").read_text())
MIN_STOCKS_PER_SIDE = clean_log["parameters"]["MIN_STOCKS_PER_SIDE"]

log = {"parameters": dict(SEED=SEED, PRESAMPLE_END=PRESAMPLE_END, AR_MAX_LAGS=AR_MAX_LAGS,
                          AR_CRITERION=AR_CRITERION, NW_LAGS=NW_LAGS, WINSOR_Q=WINSOR_Q,
                          PARIS_SPLIT=PARIS_SPLIT, OOS_FIRST_YEAR=OOS_FIRST_YEAR, RF_PARAMS=RF_PARAMS)}
print("Results go to:", RES)

Results go to: /Users/RafaelMachadoCerqueira/Desktop/Data/climate-concerns-green-brown/results


## 1. Load the cleaned data

In [2]:
read = lambda name: pd.read_csv(PROC / name, index_col=0, parse_dates=True)
panel_eu, panel_us = read("panel_eu.csv"), read("panel_us.csv")
mccc_eu, mccc_us = read("mccc_eu_calendar_full.csv"), read("mccc_us_calendar_full.csv")
stock_rets = read("stock_returns_eur.csv")
universe = pd.read_csv(RAW / "universe.csv").set_index("ticker")
GREEN = universe.index[universe["group"] == "green_eu"].tolist()
BROWN = universe.index[universe["group"] == "brown_eu"].tolist()

print(f"Europe: {panel_eu.shape}, U.S.: {panel_us.shape}")
print(f"MCCC on the European calendar: {mccc_eu.index.min().date()} -> {mccc_eu.index.max().date()}")

Europe: (3948, 50), U.S.: (3896, 47)
MCCC on the European calendar: 2003-01-01 -> 2025-06-30


## 2. Concern shocks

Investors react to **news**, not to what they already expected. Following Ardia et al., the shock is the part of today's index that an autoregressive model of its own past does not predict:

$$MCCC_t = \alpha + \sum_{k=1}^{p} \beta_k \, MCCC_{t-k} + \varepsilon_t, \qquad \text{shock}_t = \hat\varepsilon_t$$

* The lag order $p$ is chosen by `AR_CRITERION` on the pre-sample 2003–2009 only.
* **No look-ahead:** for each year from 2010, the model is re-estimated on all data *before* 1 January of that year, then used to predict that year one day ahead.
* The shocks are divided by their standard deviation, so a coefficient reads as *the effect of a one-standard-deviation surprise*.
* The index is built separately on the European and the U.S. trading calendars (02_clean).

In [3]:
def lag_matrix(y, p):
    return sm.add_constant(pd.concat({f"lag{k}": y.shift(k) for k in range(1, p + 1)}, axis=1))

def select_lag_order(y, max_lags, criterion):
    """Information criterion for p = 1..max_lags, all on the same sample."""
    X_all = lag_matrix(y, max_lags)
    scores = {}
    for p in range(1, max_lags + 1):
        d = pd.concat([y.rename("y"), X_all.iloc[:, : p + 1]], axis=1).dropna()
        scores[p] = getattr(sm.OLS(d["y"], d.drop(columns="y")).fit(), criterion)
    scores = pd.Series(scores, name=criterion)
    return int(scores.idxmin()), scores

def expanding_shocks(y, p, first_year, last_year):
    """One-day-ahead prediction errors; the model for year Y only sees data before Y."""
    d = pd.concat([y.rename("y"), lag_matrix(y, p)], axis=1).dropna()
    out = []
    for year in range(first_year, last_year + 1):
        train, test = d[d.index.year < year], d[d.index.year == year]
        if len(test):
            fit = sm.OLS(train["y"], train.drop(columns="y")).fit()
            out.append(test["y"] - fit.predict(test.drop(columns="y")))
    return pd.concat(out)

def build_shocks(level, label):
    p, scores = select_lag_order(level.loc[:PRESAMPLE_END], AR_MAX_LAGS, AR_CRITERION)
    first, last = pd.Timestamp(PRESAMPLE_END).year + 1, level.index.max().year
    raw = expanding_shocks(level, p, first, last)
    shock = raw / raw.std()
    log[f"ar_lag_order_{label}"] = p
    return shock, p, scores

shock_eu, p_eu, scores_eu = build_shocks(mccc_eu["mccc_Aggregate"], "eu")
shock_us, p_us, scores_us = build_shocks(mccc_us["mccc_Aggregate"], "us")
print(f"AR lag order chosen by {AR_CRITERION.upper()}: Europe calendar p = {p_eu}, U.S. calendar p = {p_us}")
pd.DataFrame({"Europe calendar": scores_eu, "U.S. calendar": scores_us}).round(1)

AR lag order chosen by BIC: Europe calendar p = 7, U.S. calendar p = 7


,Europe calendar,U.S. calendar
1,363.4,363.4
2,214.9,214.9
3,168.8,168.8
4,109.3,109.3
5,88.0,88.0
6,90.7,90.7
7,84.3,84.3
8,88.3,88.3
9,94.0,94.0
10,95.3,95.3


**Checks on the index and the shocks.** The ADF test asks whether the index level is stationary (an AR model on a non-stationary series would be misleading). The shocks should look like noise: little autocorrelation left, and a correlation with the index level that is far from 1.

In [4]:
level = mccc_eu["mccc_Aggregate"]
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)   # statsmodels announces a change of return type
    adf_stat, adf_p, *_ = adfuller(level.dropna(), autolag="AIC")
lb = acorr_ljungbox(shock_eu.dropna(), lags=[5, 10])

shock_check = pd.Series({
    "ADF statistic (index level)": round(adf_stat, 2),
    "ADF p-value": round(adf_p, 4),
    "Ljung-Box p-value, shocks, 5 lags": round(lb["lb_pvalue"].iloc[0], 4),
    "Ljung-Box p-value, shocks, 10 lags": round(lb["lb_pvalue"].iloc[1], 4),
    "Correlation shock / level": round(shock_eu.corr(level.reindex(shock_eu.index)), 3),
    "Shocks above +2 sd (% of days)": round((shock_eu > 2).mean() * 100, 2),
    "Shocks below -2 sd (% of days)": round((shock_eu < -2).mean() * 100, 2),
})
log["adf_pvalue_level"] = float(adf_p)
shock_check

ADF statistic (index level)          -3.6900
ADF p-value                           0.0042
Ljung-Box p-value, shocks, 5 lags     0.0931
Ljung-Box p-value, shocks, 10 lags    0.0000
Correlation shock / level             0.7220
Shocks above +2 sd (% of days)        4.5300
Shocks below -2 sd (% of days)        0.7600
dtype: float64

In [5]:
# The ten largest positive surprises: do they match known climate news days?
top = shock_eu.nlargest(10).rename("shock (sd)").to_frame()
top["index level"] = level.reindex(top.index)
top.round(2)

,shock (sd),index level
date,,
2022-03-01,5.82,3.05
2024-01-24,4.84,2.53
2024-07-02,4.78,2.84
2015-05-21,4.74,2.26
2018-10-09,4.73,2.32
2020-10-14,4.72,2.45
2018-11-27,4.62,2.55
2014-05-07,4.62,2.17
2010-07-21,4.54,1.95


## 2b. Improved concern shocks

The checks above show that the AR shocks are **not clean surprises**: strong autocorrelation is left (Ljung-Box), they track the index level, their mean is above zero and large positive values are far more frequent than large negative ones. Two likely causes:

* **Day of the week.** Monday's value averages the weekend (`mccc_n_calendar_days` counts the calendar days behind each trading-day value), so its level and its variance differ from other days.
* **Trend.** Concern rises over the sample, so a model fitted on past years under-predicts the next year and the shocks are positive on average.

Three definitions are built on each calendar (Europe, U.S.). All have **no look-ahead** and are divided by their standard deviation:

| Definition | Expected concern |
|---|---|
| `ar` | the AR(p) of section 2 (unchanged, for comparison) |
| `ar_adjusted` | AR(p) + day-of-week dummies (Monday = reference) + `mccc_n_calendar_days` + linear trend (years since the first observation); $p$ chosen by `AR_CRITERION` on 2003–2009 **with the extra regressors included**; re-estimated each year on data before 1 January of that year |
| `ma20` | mean of the previous 20 trading days; shock = (today − that mean) / standard deviation of those 20 days |

**Decision rule (fixed before any regression is run).** The main definition for the rest of the notebook is `ar_adjusted` if, **on both calendars**, (i) its Ljung-Box p-value at 10 lags is higher than that of `ar`, and (ii) its mean is closer to 0 than that of `ar`. Otherwise `ar` is kept. `ma20` is reported for comparison only. The rule is applied in code below and its outcome is stored in `log["main_shock_definition"]`.

In [6]:
# Generalised AR with extra regressors known on day t; with exog=None it is the model of section 2
def ar_design(y, p, exog=None):
    X = lag_matrix(y, p)
    return X if exog is None else pd.concat([X, exog.reindex(y.index)], axis=1)

def select_lag_order_exog(y, max_lags, criterion, exog=None):
    """Same as select_lag_order, with the extra regressors in every candidate model."""
    X_all = ar_design(y, max_lags, exog)
    extra = [] if exog is None else list(exog.columns)
    scores = {}
    for p in range(1, max_lags + 1):
        cols = ["const"] + [f"lag{k}" for k in range(1, p + 1)] + extra
        d = pd.concat([y.rename("y"), X_all[cols]], axis=1).dropna()
        scores[p] = getattr(sm.OLS(d["y"], d.drop(columns="y")).fit(), criterion)
    scores = pd.Series(scores, name=criterion)
    return int(scores.idxmin()), scores

def expanding_shocks_exog(y, p, first_year, last_year, exog=None):
    """One-day-ahead prediction errors; the model for year Y only sees data before Y."""
    d = pd.concat([y.rename("y"), ar_design(y, p, exog)], axis=1).dropna()
    out = []
    for year in range(first_year, last_year + 1):
        train, test = d[d.index.year < year], d[d.index.year == year]
        if len(test):
            fit = sm.OLS(train["y"], train.drop(columns="y")).fit()
            out.append(test["y"] - fit.predict(test.drop(columns="y")))
    return pd.concat(out)

def calendar_regressors(frame):
    """Day-of-week dummies (Monday = reference), calendar days behind the value, linear trend in years."""
    idx = frame.index
    X = pd.DataFrame({day: (idx.dayofweek == k).astype(float)
                      for k, day in enumerate(["tue", "wed", "thu", "fri"], start=1)}, index=idx)
    X["n_calendar_days"] = frame["mccc_n_calendar_days"]
    X["trend_years"] = (idx - idx[0]).days / 365.25
    return X

FIRST_YEAR = pd.Timestamp(PRESAMPLE_END).year + 1
MA_WINDOW = 20
SHOCK_DEFS = ["ar", "ar_adjusted", "ma20"]

def make_shocks(frame, col, label, definition):
    """Standardised shocks of frame[col] under one of SHOCK_DEFS (no look-ahead)."""
    y = frame[col]
    if definition == "ar":
        return build_shocks(y, label)[0]
    if definition == "ar_adjusted":
        exog = calendar_regressors(frame)
        p, _ = select_lag_order_exog(y.loc[:PRESAMPLE_END], AR_MAX_LAGS, AR_CRITERION, exog)
        raw = expanding_shocks_exog(y, p, FIRST_YEAR, y.index.max().year, exog)
        log[f"ar_adjusted_lag_order_{label}"] = p
    elif definition == "ma20":
        past = y.shift(1).rolling(MA_WINDOW)   # the 20 trading days before t, never t itself
        raw = ((y - past.mean()) / past.std()).loc[str(FIRST_YEAR):].dropna()
    else:
        raise ValueError(definition)
    return raw / raw.std()

# Sanity check: without extra regressors the generalised code gives back the section 2 shocks
y_eu = mccc_eu["mccc_Aggregate"]
assert select_lag_order_exog(y_eu.loc[:PRESAMPLE_END], AR_MAX_LAGS, AR_CRITERION)[0] == p_eu
raw_check = expanding_shocks_exog(y_eu, p_eu, FIRST_YEAR, y_eu.index.max().year)
assert np.allclose(raw_check / raw_check.std(), shock_eu)

CALENDARS = {"Europe": ("eu", mccc_eu, shock_eu), "U.S.": ("us", mccc_us, shock_us)}
shocks = {}
for cal, (label, frame, ar_shock) in CALENDARS.items():
    shocks[(cal, "ar")] = ar_shock
    for defn in ["ar_adjusted", "ma20"]:
        shocks[(cal, defn)] = make_shocks(frame, "mccc_Aggregate", label, defn)

print(f"AR lag order with the extra regressors ({AR_CRITERION.upper()}, 2003-2009): "
      f"Europe calendar p = {log['ar_adjusted_lag_order_eu']}, U.S. calendar p = {log['ar_adjusted_lag_order_us']}")

# What the extra regressors pick up: their coefficients in the pre-sample model
presample_coefs = {}
for cal, (label, frame, _) in CALENDARS.items():
    y, exog = frame["mccc_Aggregate"], calendar_regressors(frame)
    d = pd.concat([y.rename("y"), ar_design(y, log[f"ar_adjusted_lag_order_{label}"], exog)], axis=1)
    d = d.loc[:PRESAMPLE_END].dropna()
    presample_coefs[f"{cal} calendar"] = sm.OLS(d["y"], d.drop(columns="y")).fit().params[exog.columns]
pd.DataFrame(presample_coefs).round(4)

AR lag order with the extra regressors (BIC, 2003-2009): Europe calendar p = 5, U.S. calendar p = 5


,Europe calendar,U.S. calendar
tue,0.0536,0.0536
wed,0.0314,0.0314
thu,0.0380,0.0380
fri,0.0141,0.0141
n_calendar_days,-0.0354,-0.0354
trend_years,0.0283,0.0283


In [7]:
def shock_stats(s, level):
    s = s.dropna()
    lb = acorr_ljungbox(s, lags=[5, 10])
    return {"days": len(s), "mean": s.mean(),
            "LB_Q_10": lb["lb_stat"].iloc[1],
            "LB_p_5": lb["lb_pvalue"].iloc[0], "LB_p_10": lb["lb_pvalue"].iloc[1],
            "corr_level": s.corr(level.reindex(s.index)),
            "pct_above_2sd": (s > 2).mean() * 100, "pct_below_minus2sd": (s < -2).mean() * 100}

shock_table = pd.DataFrame({(cal, defn): shock_stats(shocks[(cal, defn)], CALENDARS[cal][1]["mccc_Aggregate"])
                            for cal in CALENDARS for defn in SHOCK_DEFS}).T
shock_table.index.names = ["calendar", "definition"]
with pd.option_context("display.float_format", "{:.3g}".format):
    display(shock_table)

days   mean  LB_Q_10   LB_p_5  LB_p_10  corr_level  \
calendar definition                                                           
Europe   ar          3.95e+03  0.161     46.6   0.0931  1.1e-06       0.722   
         ar_adjusted 3.95e+03 0.0195     44.1     0.27 3.19e-06       0.763   
         ma20        3.95e+03 0.0271      334 2.03e-67 9.44e-66       0.668   
U.S.     ar           3.9e+03   0.16     40.1    0.281 1.62e-05       0.723   
         ar_adjusted  3.9e+03 0.0174     27.9    0.679  0.00186       0.763   
         ma20         3.9e+03 0.0268      351 2.63e-71 2.63e-69       0.669   

                      pct_above_2sd  pct_below_minus2sd  
calendar definition                                      
Europe   ar                    4.53                0.76  
         ar_adjusted           3.85               0.912  
         ma20                  3.62                0.76  
U.S.     ar                     4.7               0.744  
         ar_adjusted           3.82               0.821  
         ma20                  3.64               0.744

**How to read it.** A clean surprise has a mean near 0, high Ljung-Box p-values (no autocorrelation left; `LB_Q_10` is the test statistic, useful when p-values are all close to 0), a low correlation with the index level, and roughly as many days above +2 sd as below −2 sd (about 2.3% each under normality, more with fat tails). A definition that improves on these lines removes part of the predictable component that the simple AR model leaves in the shocks.

In [8]:
# Decision rule stated above, applied mechanically
def rule_holds(cal):
    a, b = shock_table.loc[(cal, "ar")], shock_table.loc[(cal, "ar_adjusted")]
    return bool(b["LB_p_10"] > a["LB_p_10"] and abs(b["mean"]) < abs(a["mean"]))

rule = {cal: rule_holds(cal) for cal in CALENDARS}
MAIN_SHOCK = "ar_adjusted" if all(rule.values()) else "ar"
log["main_shock_definition"] = MAIN_SHOCK
log["main_shock_rule_holds"] = rule

# From here on, shock_eu and shock_us are the main definition
shock_eu, shock_us = shocks[("Europe", MAIN_SHOCK)], shocks[("U.S.", MAIN_SHOCK)]
print("Rule holds:", rule)
print("Main shock definition:", MAIN_SHOCK)

Rule holds: {'Europe': True, 'U.S.': True}
Main shock definition: ar_adjusted


## 3. Timing

European markets close around 17:30 CET, which is 11:30 in New York, before much of the day's U.S. news is out. A U.S. news shock dated *t* can therefore only reach European prices on day *t + 1*. Both are used:

* `shock_t`: shock on the same date; `shock_lag1`: shock on the previous trading day.
* The VIX is measured at the U.S. close, **after** the European close, so Europe uses the **previous day's** VIX change.

Controls: the market factor (Fama-French, local to each market), the Brent return, and the VIX change.

In [9]:
def add_timing(panel, shock):
    d = panel.copy()
    d["shock_t"] = shock.reindex(d.index)
    d["shock_lag1"] = shock.shift(1).reindex(d.index)
    d["vix_chg_lag1"] = d["vix_chg"].shift(1)
    return d.loc[shock.index.min():]

eu = add_timing(panel_eu, shock_eu)
us = add_timing(panel_us, shock_us)

SHOCKS = ["shock_t", "shock_lag1"]
CTRL_EU = ["ff_mkt_rf", "brent_ret", "vix_chg_lag1"]
CTRL_US = ["ff_mkt_rf", "brent_ret", "vix_chg"]
log.update(CTRL_EU=CTRL_EU, CTRL_US=CTRL_US)

print(f"Shock definition: {MAIN_SHOCK}")
print(f"Europe: {eu[['spread'] + SHOCKS + CTRL_EU].dropna().shape[0]} usable days")
print(f"U.S.:   {us[['spread'] + SHOCKS + CTRL_US].dropna().shape[0]} usable days")
eu[["spread"] + SHOCKS + CTRL_EU].describe().T.round(4)

Shock definition: ar_adjusted
Europe: 3800 usable days
U.S.:   3854 usable days


,count,mean,std,min,25%,50%,75%,max
spread,3947.0,0.0002,0.0135,-0.1148,-0.0072,-0.0001,0.0074,0.1152
shock_t,3948.0,0.0195,1.0001,-3.5615,-0.6461,-0.0880,0.5779,5.6698
shock_lag1,3948.0,0.0197,1.0001,-3.5615,-0.6459,-0.0874,0.5779,5.6698
ff_mkt_rf,3948.0,0.0003,0.0114,-0.1200,-0.0052,0.0006,0.0062,0.0854
brent_ret,3880.0,0.0003,0.0265,-0.4747,-0.0106,0.0006,0.0115,0.5099
vix_chg_lag1,3867.0,-0.0010,1.8883,-18.7100,-0.7400,-0.1000,0.5700,24.8600


## 4. Regressions

$$\text{spread}_t = a + b_0 \, \text{shock}_t + b_1 \, \text{shock}_{t-1} + \gamma' \, \text{controls}_t + u_t$$

* **Newey-West** standard errors (`NW_LAGS` lags) allow for autocorrelation and changing variance in daily returns.
* Shocks: the main definition chosen in section 2b (`MAIN_SHOCK`).
* Shock coefficients are reported in **basis points of daily spread per one-standard-deviation shock**.
* **U.S. first** (ICLN − XLE): a positive $b_0$ reproduces Ardia et al. and checks the pipeline. **Then Europe**: the answer to the question.

In [10]:
def fit_ols(df, y, X, nw_lags=NW_LAGS):
    d = df[[y] + X].dropna()
    return sm.OLS(d[y], sm.add_constant(d[X])).fit(cov_type="HAC", cov_kwds={"maxlags": nw_lags})

def shock_row(res, name):
    row = {"model": name, "n": int(res.nobs), "R2_%": round(res.rsquared * 100, 2)}
    for v in SHOCKS:
        if v in res.params:
            row[f"{v}_bps"] = round(res.params[v] * 1e4, 2)
            row[f"{v}_t"] = round(res.tvalues[v], 2)
            row[f"{v}_p"] = round(res.pvalues[v], 3)
    return row

SPECS = {
    "US (1) shock_t":                   (us, ["shock_t"]),
    "US (2) shock_t + controls":        (us, ["shock_t"] + CTRL_US),
    "US (3) both shocks + controls":    (us, SHOCKS + CTRL_US),
    "EU (1) both shocks":               (eu, SHOCKS),
    "EU (2) both shocks + controls":    (eu, SHOCKS + CTRL_EU),
}
fits = {name: fit_ols(df, "spread", X) for name, (df, X) in SPECS.items()}
main_table = pd.DataFrame([shock_row(r, n) for n, r in fits.items()]).set_index("model")
main_table

,n,R2_%,shock_t_bps,shock_t_t,shock_t_p,shock_lag1_bps,shock_lag1_t,shock_lag1_p
model,,,,,,,,
US (1) shock_t,3895,0.01,-1.81,-0.66,0.509,NaN,NaN,NaN
US (2) shock_t + controls,3854,7.78,-0.55,-0.21,0.834,NaN,NaN,NaN
US (3) both shocks + controls,3854,7.78,-0.54,-0.21,0.837,-0.62,-0.23,0.818
EU (1) both shocks,3947,0.00,0.34,0.16,0.873,0.33,0.14,0.892
EU (2) both shocks + controls,3800,4.91,0.77,0.35,0.723,0.05,0.02,0.984


In [11]:
MAIN_EU, MAIN_US = "EU (2) both shocks + controls", "US (3) both shocks + controls"
print(fits[MAIN_EU].summary())

                            OLS Regression Results                            
Dep. Variable:                 spread   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.048
Method:                 Least Squares   F-statistic:                     13.92
Date:                Mon, 02 Nov 2026   Prob (F-statistic):           1.66e-13
Time:                        18:29:50   Log-Likelihood:                 11034.
No. Observations:                3800   AIC:                        -2.206e+04
Df Residuals:                    3794   BIC:                        -2.202e+04
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0002      0.000      1.115   

**How to read it.** `shock_t_bps = 5` would mean that on a day when climate concern rises one standard deviation more than expected, the green portfolio beats the brown one by 0.05 percentage points. Compare with the daily volatility of the spread (about 1.4%) to judge the size: daily returns are mostly noise, so a low R² is expected and is not a failure.

## 4b. Regressions with each shock definition

The main U.S. model (both shocks + `CTRL_US`) and the main European model (both shocks + `CTRL_EU`) are re-run with each of the three shock definitions of section 2b. Same Newey-West errors, same units (basis points per one-standard-deviation shock).

In [12]:
MARKETS = {"U.S.": (panel_us, CTRL_US), "Europe": (panel_eu, CTRL_EU)}
rows = []
for market, (panel, ctrl) in MARKETS.items():
    for defn in SHOCK_DEFS:
        res = fit_ols(add_timing(panel, shocks[(market, defn)]), "spread", SHOCKS + ctrl)
        rows.append({"market": market, **shock_row(res, defn)})
by_definition = (pd.DataFrame(rows).rename(columns={"model": "definition"})
                 .set_index(["market", "definition"]))

# The main-definition rows are the main models of section 4
assert by_definition.loc[("Europe", MAIN_SHOCK)].equals(main_table.loc[MAIN_EU].rename(("Europe", MAIN_SHOCK)))
assert by_definition.loc[("U.S.", MAIN_SHOCK)].equals(main_table.loc[MAIN_US].rename(("U.S.", MAIN_SHOCK)))
by_definition

n  R2_%  shock_t_bps  shock_t_t  shock_t_p  \
market definition                                                   
U.S.   ar           3854  7.78         0.42       0.16      0.872   
       ar_adjusted  3854  7.78        -0.54      -0.21      0.837   
       ma20         3854  7.78        -1.03      -0.41      0.685   
Europe ar           3800  4.91         0.89       0.40      0.687   
       ar_adjusted  3800  4.91         0.77       0.35      0.723   
       ma20         3800  4.92         0.86       0.41      0.683   

                    shock_lag1_bps  shock_lag1_t  shock_lag1_p  
market definition                                               
U.S.   ar                    -0.49         -0.18         0.857  
       ar_adjusted           -0.62         -0.23         0.818  
       ma20                  -0.40         -0.16         0.872  
Europe ar                     1.02          0.42         0.673  
       ar_adjusted            0.05          0.02         0.984  
       ma20                   1.31          0.59         0.554

## 4c. Closer to Ardia et al.: the U.S. on their sample period

U.S. models (1) to (3) of section 4 re-run on the sample of Ardia et al., **1 January 2010 – 30 June 2018**, with the `ar` shocks and with the main definition.

*Caveat:* Ardia et al. sort S&P 500 firms by greenness (greenhouse-gas intensity) into green and brown portfolios; here the U.S. spread is ICLN (clean energy ETF) minus XLE (energy sector ETF). The two portfolios differ in composition and breadth, so the comparison is approximate.

In [13]:
ARDIA_START, ARDIA_END = "2010-01-01", "2018-06-30"
log["parameters"].update(ARDIA_START=ARDIA_START, ARDIA_END=ARDIA_END)

rows = []
for defn in dict.fromkeys(["ar", MAIN_SHOCK]):   # one pass only if the main definition is "ar"
    d = add_timing(panel_us, shocks[("U.S.", defn)]).loc[ARDIA_START:ARDIA_END]
    for name, (_, X) in SPECS.items():
        if name.startswith("US"):
            rows.append({"definition": defn, **shock_row(fit_ols(d, "spread", X), name)})
ardia = pd.DataFrame(rows).set_index(["definition", "model"])
ardia

n  R2_%  shock_t_bps  shock_t_t  \
definition  model                                                               
ar          US (1) shock_t                 2137  0.01        -1.52      -0.49   
            US (2) shock_t + controls      2124  3.09        -1.22      -0.39   
            US (3) both shocks + controls  2124  3.13        -1.21      -0.38   
ar_adjusted US (1) shock_t                 2137  0.00        -0.64      -0.21   
            US (2) shock_t + controls      2124  3.08        -0.36      -0.11   
            US (3) both shocks + controls  2124  3.12        -0.37      -0.12   

                                           shock_t_p  shock_lag1_bps  \
definition  model                                                      
ar          US (1) shock_t                     0.626             NaN   
            US (2) shock_t + controls          0.699             NaN   
            US (3) both shocks + controls      0.701            2.79   
ar_adjusted US (1) shock_t                     0.837             NaN   
            US (2) shock_t + controls          0.908             NaN   
            US (3) both shocks + controls      0.905            2.85   

                                           shock_lag1_t  shock_lag1_p  
definition  model                                                      
ar          US (1) shock_t                          NaN           NaN  
            US (2) shock_t + controls               NaN           NaN  
            US (3) both shocks + controls          0.93         0.353  
ar_adjusted US (1) shock_t                          NaN           NaN  
            US (2) shock_t + controls               NaN           NaN  
            US (3) both shocks + controls          0.96         0.335

## 5. Checking the assumptions

1. **Autocorrelation** of residuals (Ljung-Box) and **changing variance** (Breusch-Pagan): if present, ordinary standard errors are wrong, which is why Newey-West is used.
2. **Normality** (Jarque-Bera): daily returns have fat tails; with ~3,800 days the coefficients remain valid, but a few extreme days can drive them (see 3 and the winsorised robustness check).
3. **Influential days**: the largest residuals, to see what the model gets wrong.
4. **Linearity**: does the spread respond more than proportionally to large shocks?

In [14]:
def diagnostics(res, name):
    lb = acorr_ljungbox(res.resid, lags=[5])
    bp = het_breuschpagan(res.resid, res.model.exog)
    jb, jb_p, skew, kurt = jarque_bera(res.resid)
    return pd.Series({"Ljung-Box p (5 lags)": lb["lb_pvalue"].iloc[0], "Breusch-Pagan p": bp[1],
                      "Jarque-Bera p": jb_p, "skewness": skew, "kurtosis": kurt}, name=name)

diag = pd.concat([diagnostics(fits[MAIN_US], "U.S."), diagnostics(fits[MAIN_EU], "Europe")], axis=1).round(4)
diag

,U.S.,Europe
Ljung-Box p (5 lags),0.0036,0.0000
Breusch-Pagan p,0.2530,0.0015
Jarque-Bera p,0.0000,0.0000
skewness,-0.1075,0.3907
kurtosis,7.7303,9.2480


In [15]:
# What the model gets wrong: the ten days with the largest residuals (Europe)
res_eu = fits[MAIN_EU]
worst = res_eu.resid.abs().nlargest(10).index
pd.DataFrame({"spread_%": eu.loc[worst, "spread"] * 100, "fitted_%": res_eu.fittedvalues[worst] * 100,
              "shock_t": eu.loc[worst, "shock_t"], "brent_%": eu.loc[worst, "brent_ret"] * 100,
              "market_%": eu.loc[worst, "ff_mkt_rf"] * 100}).round(2)

,spread_%,fitted_%,shock_t,brent_%,market_%
date,,,,,
2022-02-24,11.52,0.00,-0.90,2.01,-3.56
2022-02-28,11.04,-0.41,0.88,4.59,-0.50
2020-11-09,-11.48,-0.93,0.02,7.48,3.19
2011-03-15,8.20,0.31,-0.58,-1.63,-2.23
2022-03-07,6.63,-0.30,-1.15,4.17,-2.17
2011-03-14,7.03,0.15,-0.67,-0.98,-0.40
2024-11-06,-6.60,0.23,0.51,-0.60,-2.31
2025-04-03,7.47,0.71,-1.27,-6.12,-0.89
2020-11-24,-6.73,-0.42,0.12,3.62,1.26


In [16]:
# Linearity 1: average spread by quintile of the same-day and previous-day shock (basis points)
def by_quintile(df, shock):
    d = df[["spread", shock]].dropna()
    q = pd.qcut(d[shock], 5, labels=["Q1 (most negative)", "Q2", "Q3", "Q4", "Q5 (most positive)"])
    g = d.groupby(q, observed=True)["spread"]
    return pd.DataFrame({"mean_spread_bps": g.mean() * 1e4, "days": g.size()}).round(2)

pd.concat({"shock_t": by_quintile(eu, "shock_t"), "shock_lag1": by_quintile(eu, "shock_lag1")}, axis=1)

shock_t           shock_lag1     
                   mean_spread_bps days mean_spread_bps days
Q1 (most negative)            7.17  790            0.46  790
Q2                            4.32  789            2.20  789
Q3                           -0.41  789            5.91  789
Q4                           -5.25  789            1.82  789
Q5 (most positive)            6.17  790            1.62  790

In [17]:
# Linearity 2: add squared shocks to the main European model
eu_sq = eu.assign(shock_t_sq=eu["shock_t"] ** 2, shock_lag1_sq=eu["shock_lag1"] ** 2)
res_sq = fit_ols(eu_sq, "spread", SHOCKS + ["shock_t_sq", "shock_lag1_sq"] + CTRL_EU)
res_sq.summary2().tables[1].loc[["shock_t_sq", "shock_lag1_sq"]].round(5)

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
shock_t_sq,0.00027,0.00013,2.11355,0.03455,0.00002,0.00053
shock_lag1_sq,-0.00005,0.00016,-0.28367,0.77666,-0.00037,0.00028


## 6. Robustness

Each line re-runs the main European model with one change. The question is whether the sign and rough size of the shock coefficients survive.

| Check | Change |
|---|---|
| Liquidity 5% | stricter liquidity rule (`spread_liq_robust`) |
| Local currency | returns in each stock's own currency |
| Winsorised | all variables clipped at the 1st and 99th percentiles (oil crash, Covid) |
| Without 2020 | drops the Covid year |
| Before / after Paris | 2010–2015 vs 2016–2025 |
| Fama-French 3 factors | adds size and value factors |
| Leave-one-out | rebuilds the spread without each stock in turn |

The weekend rule of the MCCC (`WEEKEND_RULE` in `02_clean`) can also be changed there and this notebook re-run.

In [18]:
def winsorise(df, cols, q):
    d = df.copy()
    for c in cols:
        lo, hi = d[c].quantile([q, 1 - q])
        d[c] = d[c].clip(lo, hi)
    return d

X_MAIN = SHOCKS + CTRL_EU
robust_specs = {
    "Main":                   (eu, "spread", X_MAIN),
    "Liquidity rule 5%":      (eu, "spread_liq_robust", X_MAIN),
    "Local currency":         (eu, "spread_local_ccy", X_MAIN),
    "Winsorised 1%":          (winsorise(eu, ["spread"] + X_MAIN, WINSOR_Q), "spread", X_MAIN),
    "Without 2020":           (eu[eu.index.year != 2020], "spread", X_MAIN),
    "Before Paris (2010-15)": (eu[eu.index < PARIS_SPLIT], "spread", X_MAIN),
    "After Paris (2016-25)":  (eu[eu.index >= PARIS_SPLIT], "spread", X_MAIN),
    "Fama-French 3 factors":  (eu, "spread", X_MAIN + ["ff_smb", "ff_hml"]),
}
robustness = pd.DataFrame([shock_row(fit_ols(df, y, X), name) for name, (df, y, X) in robust_specs.items()]).set_index("model")
robustness

,n,R2_%,shock_t_bps,shock_t_t,shock_t_p,shock_lag1_bps,shock_lag1_t,shock_lag1_p
model,,,,,,,,
Main,3800,4.91,0.77,0.35,0.723,0.05,0.02,0.984
Liquidity rule 5%,3799,3.78,0.86,0.38,0.701,0.46,0.19,0.853
Local currency,3800,4.91,0.86,0.40,0.689,0.29,0.12,0.902
Winsorised 1%,3800,5.91,1.79,0.85,0.395,-0.41,-0.20,0.844
Without 2020,3554,4.86,1.36,0.63,0.527,-1.40,-0.59,0.554
Before Paris (2010-15),1450,3.47,-2.17,-0.69,0.489,3.97,1.42,0.157
After Paris (2016-25),2350,5.60,2.16,0.77,0.443,-1.89,-0.59,0.553
Fama-French 3 factors,3800,18.82,1.56,0.77,0.440,0.81,0.37,0.712


In [19]:
# Leave-one-out: rebuild the spread without each stock, same rule as 02_clean (min stocks per side)
def side_mean(rets, members):
    r = rets[members]
    return r.mean(axis=1).where(r.notna().sum(axis=1) >= MIN_STOCKS_PER_SIDE)

def spread_without(drop=None):
    g = [t for t in GREEN if t != drop]
    b = [t for t in BROWN if t != drop]
    return side_mean(stock_rets, g) - side_mean(stock_rets, b)

# Sanity check: with every stock, the rebuilt spread equals the one from 02_clean
gap = (spread_without() - panel_eu["spread"]).abs().max()
assert gap < 1e-10, f"Rebuilt spread differs from panel_eu by {gap}"

loo_rows = []
for t in GREEN + BROWN:
    d = eu.copy()
    d["spread"] = spread_without(t).reindex(d.index)
    row = shock_row(fit_ols(d, "spread", X_MAIN), t)
    row["side"] = "green" if t in GREEN else "brown"
    loo_rows.append(row)
leave_one_out = pd.DataFrame(loo_rows).set_index("model")
print("Range of the shock coefficients when one stock is dropped (bps):")
print(leave_one_out[["shock_t_bps", "shock_lag1_bps"]].agg(["min", "median", "max"]).round(2))
leave_one_out.sort_values("shock_lag1_bps")

Range of the shock coefficients when one stock is dropped (bps):
        shock_t_bps  shock_lag1_bps
min            0.20           -0.30
median         0.77            0.04
max            1.16            0.65


,n,R2_%,shock_t_bps,shock_t_t,shock_t_p,shock_lag1_bps,shock_lag1_t,shock_lag1_p,side
model,,,,,,,,,
VWS.CO,3800,5.36,1.02,0.47,0.639,-0.30,-0.13,0.896,green
MT.AS,3800,4.43,0.92,0.43,0.671,-0.19,-0.08,0.936,brown
AF.PA,3800,5.49,0.77,0.35,0.725,-0.17,-0.07,0.941,brown
GRE.MC,3800,4.75,0.54,0.25,0.802,-0.16,-0.07,0.948,green
SLR.MC,3800,5.12,1.16,0.55,0.583,-0.15,-0.06,0.950,green
SCATC.OL,3800,5.07,0.82,0.38,0.707,-0.11,-0.05,0.962,green
LHA.DE,3800,5.53,0.70,0.32,0.750,-0.10,-0.04,0.965,brown
S92.DE,3800,5.52,0.36,0.17,0.867,-0.10,-0.04,0.966,green
BP.L,3800,4.75,0.70,0.32,0.746,-0.08,-0.03,0.974,brown


**Extension: which kind of climate news?** The MCCC comes with four thematic sub-indices (`cluster_…`). Each gets its own shocks (main definition of section 2b), and the main European model is re-run with them instead of the aggregate.

In [20]:
cluster_cols = [c for c in mccc_eu.columns if c.startswith("mccc_cluster_")]
theme_rows = []
for col in cluster_cols:
    s = make_shocks(mccc_eu, col, "eu_" + col.replace("mccc_cluster_", "").replace(" ", "_"), MAIN_SHOCK)
    row = shock_row(fit_ols(add_timing(panel_eu, s), "spread", X_MAIN), col.replace("mccc_cluster_", ""))
    theme_rows.append(row)
themes = pd.DataFrame(theme_rows).set_index("model")
themes

,n,R2_%,shock_t_bps,shock_t_t,shock_t_p,shock_lag1_bps,shock_lag1_t,shock_lag1_p
model,,,,,,,,
Business Impact,3800,4.95,2.68,1.20,0.232,-0.84,-0.38,0.701
Environmental Impact,3800,4.91,1.33,0.56,0.577,-0.14,-0.06,0.949
Societal Debate,3800,4.90,0.23,0.10,0.916,-0.13,-0.06,0.955
Research,3800,4.93,2.08,1.03,0.302,1.04,0.38,0.701


## 6b. Longer horizons

Daily returns may react to concern news only gradually, or the effect may be lost in daily noise. Two longer horizons, main shock definition, Europe and U.S.:

* **Cumulative 5-day spread:** the sum of the spread from $t$ to $t+4$ regressed on $\text{shock}_t$, $\text{shock}_{t-1}$ and the same-day controls. The 5-day windows overlap, so Newey-West uses 10 lags.
* **Weekly data:** calendar weeks ending on Friday; sums of the daily spread, of the daily shocks ($\text{shock}_t$), of the Brent return, of the market factor and of the VIX change (the previous day's change for Europe, as in the daily model). Weekly spread on weekly shock with controls, Newey-West with 4 lags. A Friday U.S. shock can only reach European prices the following Monday, which falls in the next week.

The weekly shock is a sum of standardised daily shocks, so its coefficient is the effect of one daily standard deviation of surprise during the week. The daily main models of section 4 are repeated for reference.

In [21]:
HORIZON, NW_LAGS_CUM, NW_LAGS_WEEK = 5, 10, 4
log["parameters"].update(HORIZON=HORIZON, NW_LAGS_CUM=NW_LAGS_CUM, NW_LAGS_WEEK=NW_LAGS_WEEK)

def horizon_rows(market, df, ctrl, daily_fit):
    d = df.assign(spread_cum5=df["spread"].rolling(HORIZON).sum().shift(-(HORIZON - 1)))   # t .. t+4
    cum = fit_ols(d, "spread_cum5", SHOCKS + ctrl, nw_lags=NW_LAGS_CUM)
    w = df[["spread", "shock_t"] + ctrl].dropna().resample("W-FRI").sum(min_count=1).dropna()
    week = fit_ols(w, "spread", ["shock_t"] + ctrl, nw_lags=NW_LAGS_WEEK)
    return [{"market": market, **shock_row(r, h)} for h, r in
            [("daily (section 4)", daily_fit), ("cumulative 5 days", cum), ("weekly", week)]]

horizons = pd.DataFrame(horizon_rows("U.S.", us, CTRL_US, fits[MAIN_US]) +
                        horizon_rows("Europe", eu, CTRL_EU, fits[MAIN_EU]))
horizons = horizons.rename(columns={"model": "horizon"}).set_index(["market", "horizon"])
horizons

n   R2_%  shock_t_bps  shock_t_t  shock_t_p  \
market horizon                                                             
U.S.   daily (section 4)  3854   7.78        -0.54      -0.21      0.837   
       cumulative 5 days  3850   1.44         2.56       0.41      0.684   
       weekly              809  12.24         0.41       0.07      0.943   
Europe daily (section 4)  3800   4.91         0.77       0.35      0.723   
       cumulative 5 days  3796   0.96         3.84       0.81      0.415   
       weekly              809   5.85         2.02       0.38      0.701   

                          shock_lag1_bps  shock_lag1_t  shock_lag1_p  
market horizon                                                        
U.S.   daily (section 4)           -0.62         -0.23         0.818  
       cumulative 5 days            1.83          0.31         0.757  
       weekly                        NaN           NaN           NaN  
Europe daily (section 4)            0.05          0.02         0.984  
       cumulative 5 days            4.39          0.86         0.390  
       weekly                        NaN           NaN           NaN

## 6c. Large shocks (exploratory)

The main European and U.S. models get two dummies: `large_t` = 1 when $\text{shock}_t > 2$ and `large_lag1` = 1 when $\text{shock}_{t-1} > 2$; the linear shock terms stay in.

**This test is exploratory, not confirmatory.** It was suggested by the first results (the quintile means, the squared shock terms and the random-forest partial dependence in sections 5 and 7), so it uses the same data that motivated it, and its p-values overstate the evidence. It should be read as a description to be confirmed on other data.

In [22]:
LARGE_SD = 2
log["parameters"]["LARGE_SD"] = LARGE_SD

rows = []
for market, df, ctrl in [("U.S.", us, CTRL_US), ("Europe", eu, CTRL_EU)]:
    d = df.assign(large_t=(df["shock_t"] > LARGE_SD).astype(float),
                  large_lag1=(df["shock_lag1"] > LARGE_SD).astype(float))
    res = fit_ols(d, "spread", SHOCKS + ["large_t", "large_lag1"] + ctrl)
    for v in ["large_t", "large_lag1"]:
        rows.append({"market": market, "dummy": v, "n": int(res.nobs),
                     "large_shock_days": int(res.model.data.orig_exog[v].sum()),
                     "bps": round(res.params[v] * 1e4, 2), "t": round(res.tvalues[v], 2),
                     "p": round(res.pvalues[v], 3)})
large = pd.DataFrame(rows).set_index(["market", "dummy"])
large

n  large_shock_days    bps     t      p
market dummy                                                 
U.S.   large_t     3854               148  15.99  1.05  0.295
       large_lag1  3854               147  -6.13 -0.41  0.681
Europe large_t     3800               148  19.45  1.50  0.133
       large_lag1  3800               149   5.72  0.38  0.704

## 7. Non-linear model: random forest vs linear regression, out of sample

Does a flexible model find structure the linear regression misses? Four models explain the European spread **on days they have never seen**:

| Model | Inputs |
|---|---|
| Mean | none (the historical average spread) — the benchmark |
| Linear, controls | market, Brent, VIX |
| Linear, controls + shocks | + same-day and previous-day concern shocks (main definition, section 2b) |
| Random forest | same inputs as the line above |

**Time-ordered validation:** for each test year from `OOS_FIRST_YEAR`, every model is trained on all earlier years only. Random shuffling would let the model learn from the future.

**Metric:** out-of-sample R² relative to the mean, $R^2_{OOS} = 1 - \sum (y - \hat y_{model})^2 / \sum (y - \hat y_{mean})^2$. Positive means better than the mean. The inputs include same-day market moves, so this measures **explanatory power on unseen days**, not a trading forecast.

**Random-forest settings** (`RF_PARAMS`) are fixed in advance rather than tuned: shallow trees and at least 50 days per leaf, because a flexible model on noisy daily data mostly learns noise. This also keeps the run within the Nuvolos workspace.

In [23]:
def out_of_sample(df, y, controls, first_year):
    full = controls + SHOCKS
    d = df[[y] + full].dropna()
    preds, importances = [], []
    for year in range(first_year, d.index.year.max() + 1):
        train, test = d[d.index.year < year], d[d.index.year == year]
        if test.empty:
            continue
        p = pd.DataFrame({"y": test[y], "mean": train[y].mean()}, index=test.index)
        p["linear_controls"] = LinearRegression().fit(train[controls], train[y]).predict(test[controls])
        p["linear_full"] = LinearRegression().fit(train[full], train[y]).predict(test[full])
        rf = RandomForestRegressor(**RF_PARAMS).fit(train[full], train[y])
        p["random_forest"] = rf.predict(test[full])
        preds.append(p)
        imp = permutation_importance(rf, test[full], test[y], n_repeats=5, random_state=SEED)
        importances.append(pd.Series(imp.importances_mean, index=full, name=year))
    return pd.concat(preds), pd.concat(importances, axis=1).T

def r2_oos(p, model):
    return 1 - ((p["y"] - p[model]) ** 2).sum() / ((p["y"] - p["mean"]) ** 2).sum()

MODELS = ["linear_controls", "linear_full", "random_forest"]
preds_eu, imp_eu = out_of_sample(eu, "spread", CTRL_EU, OOS_FIRST_YEAR)
preds_us, imp_us = out_of_sample(us, "spread", CTRL_US, OOS_FIRST_YEAR)

oos_summary = pd.DataFrame({
    "Europe": {m: r2_oos(preds_eu, m) * 100 for m in MODELS},
    "U.S.":   {m: r2_oos(preds_us, m) * 100 for m in MODELS},
}).round(2)
oos_summary.index.name = "out-of-sample R2 (%)"
oos_summary

,Europe,U.S.
out-of-sample R2 (%),,
linear_controls,4.34,7.76
linear_full,4.21,7.68
random_forest,4.38,6.41


In [24]:
# Year by year (Europe): is the result driven by a few years?
oos_by_year = pd.DataFrame({m: preds_eu.groupby(preds_eu.index.year).apply(lambda p: r2_oos(p, m)) * 100
                            for m in MODELS}).round(2)
oos_by_year["shocks_add_%"] = (oos_by_year["linear_full"] - oos_by_year["linear_controls"]).round(2)
oos_by_year

,linear_controls,linear_full,random_forest,shocks_add_%
date,,,,
2015,5.87,5.94,2.57,0.07
2016,19.36,19.29,11.34,-0.07
2017,9.06,8.49,6.95,-0.57
2018,-0.31,-1.14,-0.82,-0.83
2019,9.10,8.67,7.45,-0.43
2020,4.27,4.35,8.69,0.08
2021,2.02,1.88,2.77,-0.14
2022,2.01,1.77,1.35,-0.24
2023,2.71,2.69,3.18,-0.02


In [25]:
# Which inputs does the random forest use? Permutation importance on the test years (Europe):
# how much the test error rises when one input is shuffled; near zero or negative = not used
importance = pd.DataFrame({"Europe": imp_eu.mean(), "U.S.": imp_us.mean()})
# Shown as % of the total absolute importance in each market
(importance / importance.abs().sum() * 100).round(1).sort_values("Europe", ascending=False)

,Europe,U.S.
brent_ret,92.9,96.7
ff_mkt_rf,4.2,-0.1
shock_t,1.0,-1.0
vix_chg_lag1,-0.6,NaN
shock_lag1,-1.4,-0.7
vix_chg,NaN,-1.4


In [26]:
# Shape of the relationship the forest learns: partial dependence on each shock (trained on all years)
d = eu[["spread"] + CTRL_EU + SHOCKS].dropna()
rf_all = RandomForestRegressor(**RF_PARAMS).fit(d[CTRL_EU + SHOCKS], d["spread"])
pd_rows = []
for v in SHOCKS:
    pdp = partial_dependence(rf_all, d[CTRL_EU + SHOCKS], [v], grid_resolution=20)
    pd_rows.append(pd.DataFrame({"variable": v, "shock_value": pdp["grid_values"][0],
                                 "predicted_spread_bps": pdp["average"][0] * 1e4}))
partial = pd.concat(pd_rows)
# One column pair per shock: grid value and the forest's average predicted spread at that value
pd.concat({v: g.reset_index(drop=True)[["shock_value", "predicted_spread_bps"]]
           for v, g in partial.groupby("variable")}, axis=1).round(2)

shock_lag1                          shock_t                     
   shock_value predicted_spread_bps shock_value predicted_spread_bps
0        -1.41                 0.92       -1.40                 3.89
1        -1.24                 1.46       -1.23                 2.91
2        -1.07                 1.67       -1.06                 2.32
3        -0.90                 1.02       -0.90                 2.35
4        -0.73                 1.37       -0.73                 1.69
5        -0.56                 1.27       -0.56                 2.09
6        -0.39                 2.15       -0.39                 0.77
7        -0.22                 2.14       -0.22                 0.34
8        -0.05                 2.18       -0.05                 0.17
9         0.12                 2.14        0.12                -0.03
10        0.29                 1.92        0.29                -0.38
11        0.46                 1.86        0.46                -1.19
12        0.63                 1.71        0.63                -0.68
13        0.80                 1.37        0.80                -0.81
14        0.97                 1.69        0.97                 0.37
15        1.14                 1.64        1.14                 2.40
16        1.31                 1.92        1.31                 6.91
17        1.48                 1.64        1.48                 8.13
18        1.65                 1.95        1.65                 9.22
19        1.82                 2.06        1.82                 9.05

## 8. Save the results

Small summary tables only — they contain no raw data and can be committed. The daily shocks go to `data/processed/` (ignored by Git).

In [27]:
main_table.to_csv(RES / "regressions_main.csv")
pd.DataFrame({n: r.params for n, r in fits.items()}).to_csv(RES / "regressions_all_coefficients.csv")
diag.to_csv(RES / "diagnostics.csv")
robustness.to_csv(RES / "robustness.csv")
leave_one_out.to_csv(RES / "leave_one_out.csv")
themes.to_csv(RES / "thematic_subindices.csv")
oos_summary.to_csv(RES / "oos_summary.csv")
oos_by_year.to_csv(RES / "oos_by_year.csv")
importance.to_csv(RES / "rf_importance.csv")
partial.to_csv(RES / "rf_partial_dependence.csv", index=False)
shock_check.to_csv(RES / "shock_checks.csv")
shock_table.to_csv(RES / "shock_definitions.csv")
by_definition.to_csv(RES / "regressions_by_shock_definition.csv")
ardia.to_csv(RES / "us_ardia_period.csv")
horizons.to_csv(RES / "longer_horizons.csv")
large.to_csv(RES / "large_shocks.csv")
# "eu" / "us" = main definition; the other columns are every definition, for comparison
all_shocks = {"eu": shock_eu, "us": shock_us,
              **{f"{CALENDARS[cal][0]}_{defn}": s for (cal, defn), s in shocks.items()}}
pd.concat(all_shocks, axis=1, sort=True).to_csv(PROC / "concern_shocks.csv")

log["main_eu"] = main_table.loc[MAIN_EU].to_dict()
log["main_us"] = main_table.loc[MAIN_US].to_dict()
log["oos_r2_pct"] = oos_summary.to_dict()
records = lambda t: t.reset_index().to_dict(orient="records")
log["shock_definitions"] = records(shock_table)
log["regressions_by_shock_definition"] = records(by_definition)
log["us_ardia_period"] = records(ardia)
log["longer_horizons"] = records(horizons)
log["large_shocks"] = records(large)
(RES / "model_log.json").write_text(json.dumps(log, indent=2, default=str))
print("Saved:", sorted(p.name for p in RES.iterdir()))

Saved: ['diagnostics.csv', 'large_shocks.csv', 'leave_one_out.csv', 'longer_horizons.csv', 'model_log.json', 'oos_by_year.csv', 'oos_summary.csv', 'regressions_all_coefficients.csv', 'regressions_by_shock_definition.csv', 'regressions_main.csv', 'rf_importance.csv', 'rf_partial_dependence.csv', 'robustness.csv', 'shock_checks.csv', 'shock_definitions.csv', 'thematic_subindices.csv', 'us_ardia_period.csv']
